In [ ]:
!pip install ultralytics
!pip install roboflow

# Get Dataset

In [2]:
from roboflow import Roboflow

rf = Roboflow(api_key="5Yyg92eaTatOli8j55ud") #API는 가리거나 한다

project = rf.workspace("roboflow-jvuqo").project("football-players-detection-3zvbc")

version = project.version(14)

dataset = version.download("yolov8")

loading Roboflow workspace...
loading Roboflow project...
Dependency ultralytics<=8.0.20 is required but found version=8.4.59, to fix: `pip install ultralytics<=8.0.20`


Extracting Dataset Version Zip to football-players-detection-14 in yolov8:: 100%|██████████| 756/756 [00:03<00:00, 203.01it/s]


In [3]:
dataset.location

'c:\\Users\\kimwc\\Desktop\\upload - 복사본\\yolo_flask\\football-players-detection-14'

In [3]:
import shutil

shutil.move('football-players-detection-1/train',
            'football-players-detection-1/football-players-detection-1/train'
            )

shutil.move('football-players-detection-1/test',
            'football-players-detection-1/football-players-detection-1/test'
            )

shutil.move('football-players-detection-1/valid',
            'football-players-detection-1/football-players-detection-1/valid'
            )

'football-players-detection-1/football-players-detection-1/valid'

# Training

In [ ]:
!yolo task=detect mode=train model=yolov5x.pt data={dataset.location}/data.yaml epochs=100 imgsz=640

In [5]:
import yaml
import os

# 데이터셋이 위치한 실제 절대 경로
dataset_path = r"c:\Users\kimwc\Desktop\upload - 복사본\yolo_flask\football-players-detection-14"
yaml_file_path = os.path.join(dataset_path, "data.yaml")

# data.yaml 파일 읽기
with open(yaml_file_path, "r", encoding="utf-8") as f:
    yaml_data = yaml.safe_load(f)

# 윈도우 환경에 맞게 하위 경로 및 루트 경로(path) 강제 고정
yaml_data["path"] = dataset_path
yaml_data["train"] = "train/images"
yaml_data["val"] = "valid/images"
if "test" in yaml_data:
    yaml_data["test"] = "test/images"

# 수정된 데이터를 깔끔하게 덮어쓰기
with open(yaml_file_path, "w", encoding="utf-8") as f:
    yaml.dump(yaml_data, f, default_flow_style=False)

print(f"🔥 오늘 기준 data.yaml 재교정 완료!")
print(f"실제 바라보는 경로: {yaml_data['path']}")

🔥 오늘 기준 data.yaml 재교정 완료!
실제 바라보는 경로: c:\Users\kimwc\Desktop\upload - 복사본\yolo_flask\football-players-detection-14


In [11]:
from ultralytics import YOLO

model = YOLO("yolov8x.pt")

model.train(
    data=r"c:\Users\kimwc\Desktop\upload - 복사본\yolo_flask\football-players-detection-14\data.yaml", 
    epochs=100, 
    imgsz=640,
    device=0,
    amp=False,
    workers=0,
    batch=4,       # 🚀 배치를 4로 줄여 8GB VRAM 초과 현상 차단
    half=False,
    val=False,
    compile=False,
    cache=False    # RAM 선점 방지를 위해 캐시 해제
)

New https://pypi.org/project/ultralytics/8.4.60 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.59  Python-3.8.8 torch-2.4.1+cu118 CUDA:0 (NVIDIA GeForce RTX 2070, 8192MiB)
engine\trainer: agnostic_nms=False, amp=False, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=c:\Users\kimwc\Desktop\upload - \yolo_flask\football-players-detection-14\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yo

KeyboardInterrupt: 

In [12]:
import torch
import os

# 윈도우 비동기 에러를 막고, GPU 연산 병목을 하드웨어 레벨에서 강제 해결
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
os.environ["TORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:128"

# 🚀 엔비디아 그래픽카드 가속 엔진(CuDNN) 최고 성능 튜닝 활성화
torch.backends.cudnn.benchmark = True
torch.backends.cudnn.enabled = True

print("✅ 하드웨어 GPU 가속 최적화 고정 완료!")

✅ 하드웨어 GPU 가속 최적화 고정 완료!


In [13]:
from ultralytics import YOLO

# 원래 사용하시려던 무거운 x 모델 그대로 탑재
model = YOLO("yolov8x.pt")

# 물리적 GPU 하드웨어 연산 패스만 정확히 정조준하여 학습 시작
model.train(
    data=r"c:\Users\kimwc\Desktop\upload - 복사본\yolo_flask\football-players-detection-14\data.yaml", 
    epochs=100, 
    imgsz=640,
    device=0,
    amp=False,        # NotImplementedError 방지
    workers=0,        # 윈도우 멀티프로세싱 크래시 방지
    batch=8,          # RTX 2070 VRAM 안착
    val=False,        # 검증 단계 충돌 완벽 차단
    half=False,       # 반정밀도 연산 버그 우회
    compile=False     # CPU로 새는 백본 현상 차단
)

New https://pypi.org/project/ultralytics/8.4.60 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.59  Python-3.8.8 torch-2.4.1+cu118 CUDA:0 (NVIDIA GeForce RTX 2070, 8192MiB)
engine\trainer: agnostic_nms=False, amp=False, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=c:\Users\kimwc\Desktop\upload - \yolo_flask\football-players-detection-14\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yo

KeyboardInterrupt: 